## Preprocessing danych MIMIC-IV

### 📋 Cel notebooka
Ten notebook przetwarza dane rzeczywiste z MIMIC-IV, generując sekwencje tokenów reprezentujące pacjentów i ich etykiety triage.

### 🔄 Wymagania
- `data/raw/mimic/hosp/labevents.csv.gz` - wyniki badań laboratoryjnych z MIMIC-IV
- `config.py` - konfiguracja ścieżek i mapowań

### 📊 Metodologia

#### 1. Format danych wejściowych
- Plik `labevents.csv.gz` zawiera: `subject_id`, `hadm_id`, `itemid`, `valuenum`
- Dane są kompresowane (gzip) - Polars automatycznie obsługuje dekompresję

#### 2. Mapowanie MIMIC → nasze tokeny
- **itemid → nazwa testu:** Mapowanie kodów MIMIC na nasze nazwy testów (np. `HGB`, `WBC`)
- **valuenum → kwantyzacja:** Porównanie wartości z normami, generowanie tokenów Q1-Q10
- **Krytyczne wartości:** Automatyczne wykrywanie wartości poza zakresem bezpieczeństwa

#### 3. Generowanie sekwencji
- **Grupowanie:** Po `subject_id` i `hadm_id` (jeden pacjent = jedna sekwencja)
- **Tokeny:** Wiek, płeć, wyniki badań (kwantyzowane), composite features, etykiety triage
- **Composite features:** Kombinacje tokenów reprezentujące kontekst kliniczny (np. `COMPOSITE_ANEMIA_INDICATOR` = HGB niski + MCV niski)
- **Etykiety:** Mapowanie na kategorie triage na podstawie wyników badań i diagnoz

#### 4. Przetwarzanie strumieniowe
- **Metoda:** `pl.read_csv_batched` - przetwarzanie w chunkach (1M wierszy na chunk)
- **Zapis:** Tymczasowe pliki Parquet dla każdego chunka
- **Optymalizacja:** Przetwarzanie równoległe, minimalizacja użycia pamięci

### 📝 Output
- `mimic_data_YYYYMMDD_HHMMSS.txt` - przetworzone sekwencje z etykietami

In [ ]:
import sys
from pathlib import Path
import json
import polars as pl
from tqdm.auto import tqdm
import gc

# Add project root to sys.path (prepend so it wins over site-packages)
PROJECT_ROOT = Path(".").resolve().parent
project_root_str = str(PROJECT_ROOT)
if project_root_str in sys.path:
    sys.path.remove(project_root_str)
sys.path.insert(0, project_root_str)
import config

print(f"✅ Imported config from: {getattr(config, '__file__', '<unknown>')}")

# Using config paths
DATA_PATH = config.DATA_DIR
# ⚠️ UWAGA: BATCHES_DIR nie jest używany w notebooku MIMIC (tylko w notebooku 01_Synthea)
# BATCHES_DIR jest używany tylko w 01_Preprocessing_Synthea.ipynb
QUESTIONS_FILE_PATH = config.QUESTIONS_PATH
LAB_NORMS_FILE_PATH = config.LAB_NORMS_PATH
LAB_CODES = config.LAB_CODES
DIAGNOSIS_MAP = config.DIAGNOSIS_MAP
PRIORITY_MAP = config.PRIORITY_MAP
NORMS_FILE_PATH = config.LAB_NORMS_PATH
MIMIC_DIR = DATA_PATH / "raw" / "mimic" / "hosp"
MIMIC_MAPPING = config.MIMIC_MAPPING
TEMP_CHUNKS_DIR = Path("temp_chunks")
TESTS_MIN_STRATEGY = ['HGB', 'HCT', 'PLT', 'MCV']
TESTS_MAX_STRATEGY = ['WBC', 'CREATININE', 'UREA']

In [ ]:
def load_json(filepath):
    try:
        with open(filepath, "r") as f:
            return json.load(f)
    except Exception as e:
        print(f"⚠️ Nie udało się wczytać {filepath}: {e}")
        return {}

LAB_NORMS_DB = load_json(NORMS_FILE_PATH)
QUESTIONS_DB = load_json(QUESTIONS_FILE_PATH)

def get_lab_token_v2(test_name, val, age, sex, norms_db):
    if val is None: return None
    
    # 1. Pobieramy normy
    rules = norms_db.get(test_name)
    if not rules: return f"{test_name}_VAL"
    
    active_rule = None
    if isinstance(rules, list):
        for rule in rules:
            if "min_age" in rule and age < rule["min_age"]: continue
            if "max_age" in rule and age > rule["max_age"]: continue
            if "sex" in rule and rule["sex"] != sex: continue
            active_rule = rule
            break
        if not active_rule:
            active_rule = next((r for r in rules if "min_age" not in r and "sex" not in r), rules[-1])
    else:
        active_rule = rules
    
    # 2. Wyciągamy granice
    low = active_rule.get("low", -float('inf'))
    high = active_rule.get("high", float('inf'))
    crit_low = active_rule.get("critical_low", low * 0.6) 
    crit_high = active_rule.get("critical_high", high * 1.5) 
    
    # 3. KWANTYZACJA (Q1-Q10)
    token_suffix = ""
    
    if val < low:
        # Poniżej normy
        if val < crit_low:
            token_suffix = "Q1_CRITICAL"
        elif val < low * 0.85:
            token_suffix = "Q2"
        else:
            token_suffix = "Q3"
    elif val > high:
        # Powyżej normy
        if val > crit_high:
            token_suffix = "Q10_CRITICAL"
        elif val > high * 1.15:
            token_suffix = "Q9"
        else:
            token_suffix = "Q8"
    else:
        # W normie (dzielimy na 4 części)
        step = (high - low) / 4
        if val < low + step: token_suffix = "Q4"
        elif val < low + 2*step: token_suffix = "Q5"
        elif val < low + 3*step: token_suffix = "Q6"
        else: token_suffix = "Q7"
        
    return f"{test_name}_{token_suffix}"

def get_age_group(age):
    if age < 18: return "kids"
    if age < 30: return "under_30"
    if age < 60: return "under_60"
    return "seniors"

print(f"✅ Załadowano normy ({len(LAB_NORMS_DB)}) i pytania ({len(QUESTIONS_DB)}).")

In [ ]:
print("👥 Wczytywanie pacjentów i diagnoz...")

# 1. Pacjenci
patients_file = MIMIC_DIR / "patients.csv.gz"
if not patients_file.exists(): patients_file = MIMIC_DIR / "patients.csv"

df_pat = pl.read_csv(patients_file).select(["subject_id", "anchor_age", "gender"])
pat_lookup = {
    row["subject_id"]: (row["anchor_age"], row["gender"])
    for row in df_pat.iter_rows(named=True)
}

# 2. Diagnozy (ICD) - Potrzebne do symulacji wywiadu i targetu
# Wczytujemy i indeksujemy w pamięci (set kodów dla każdego pacjenta)
diag_file = MIMIC_DIR / "diagnoses_icd.csv.gz"
if not diag_file.exists(): diag_file = MIMIC_DIR / "diagnoses_icd.csv"

print("   Indeksowanie ICD...")
diag_df = pl.read_csv(diag_file).select(["subject_id", "icd_code"])

mimic_icd_map = {}
for row in tqdm(diag_df.iter_rows(named=True), total=len(diag_df), desc="ICD Indexing"):
    sid = row["subject_id"]
    code = str(row["icd_code"])
    if sid not in mimic_icd_map:
        mimic_icd_map[sid] = set()
    mimic_icd_map[sid].add(code)

del df_pat, diag_df
gc.collect()
print(f"✅ Dane gotowe. Załadowano {len(pat_lookup)} pacjentów i diagnozy.")

In [ ]:
print("🚜 Rozpoczynam przetwarzanie labevents...")

# ⚠️ WAŻNE: Plik jest spakowany jako .gz
LABEVENTS_FILE = MIMIC_DIR / "labevents.csv.gz"
if not LABEVENTS_FILE.exists():
    # Fallback: sprawdź czy jest bez .gz
    LABEVENTS_FILE = MIMIC_DIR / "labevents.csv"
    if not LABEVENTS_FILE.exists():
        raise FileNotFoundError(f"Nie znaleziono labevents.csv.gz ani labevents.csv w {MIMIC_DIR}")

print(f"📂 Wczytywanie z: {LABEVENTS_FILE}")

# Utwórz katalog dla chunków, jeśli nie istnieje
TEMP_CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Katalog tymczasowy: {TEMP_CHUNKS_DIR}")

BATCH_SIZE = 1_000_000

reader = pl.read_csv_batched(
    LABEVENTS_FILE,  # Polars automatycznie obsługuje .gz
    batch_size=BATCH_SIZE,
    schema_overrides={
        "subject_id": pl.Int32, 
        "itemid": pl.Int32, 
        "valuenum": pl.Float64,
        "hadm_id": pl.Float64
    },
    ignore_errors=True
)

batch_idx = 0
items_to_keep = list(MIMIC_MAPPING.keys())
total_rows_processed = 0

# Pasek postępu - nie znamy totala dla CSV, więc po prostu liczymy przetworzone wiersze
pbar = tqdm(desc="Przetwarzanie wierszy (mln)", unit="batch")

while True:
    batches = reader.next_batches(1)
    if not batches:
        break 

    df_chunk = batches[0]
    rows_in_chunk = len(df_chunk)
    total_rows_processed += rows_in_chunk

    # Filtrowanie
    df_filtered = (
        df_chunk
        .select(["subject_id", "hadm_id", "itemid", "valuenum"])
        .filter(pl.col("itemid").is_in(items_to_keep))
        .filter(pl.col("hadm_id").is_not_null())
        .cast({
            "hadm_id": pl.Int32,
            "valuenum": pl.Float32 
        })
    )

    if not df_filtered.is_empty():
        out_path = TEMP_CHUNKS_DIR / f"chunk_{batch_idx:04d}.parquet"
        df_filtered.write_parquet(out_path)
    
    batch_idx += 1
    
    # Aktualizacja paska postępu
    pbar.update(rows_in_chunk / 1_000_000) # Aktualizuj o ułamek miliona
    pbar.set_postfix({"chunks": batch_idx, "saved_rows": len(df_filtered)})

    del df_chunk, df_filtered
    # GC co 5 paczek (5 mln wierszy), żeby nie spowalniać za często
    if batch_idx % 5 == 0:
        gc.collect()

pbar.close()
print(f"✅ Przetworzono {total_rows_processed:,} wierszy. Zapisano chunki.")

In [ ]:
# Cell 5: Inteligentna Agregacja (Min/Max pivot)

print("🧮 Agregowanie labów (Strategia Critical Value)...")

# 1. Przygotowanie wyrażeń agregacji
# Musimy pogrupować ID kodów według nazwy testu (np. HGB ma dwa kody: 51222, 50811)
name_to_ids = {}
for code, name in MIMIC_MAPPING.items():
    name_to_ids.setdefault(name, []).append(code)

agg_exprs = []

for name, ids in name_to_ids.items():
    # Filtrujemy tylko wiersze dotyczące danego badania
    val_col = pl.col("valuenum").filter(pl.col("itemid").is_in(ids))
    
    if name in TESTS_MIN_STRATEGY:
        # Dla anemii bierzemy minimum (najgorszy wynik w trakcie pobytu)
        expr = val_col.min().alias(name)
    else:
        # Dla infekcji/nerek bierzemy maximum (najgorszy pik)
        expr = val_col.max().alias(name)
        
    agg_exprs.append(expr)

# 2. Wykonanie
labs = pl.scan_parquet(str(TEMP_CHUNKS_DIR / "*.parquet"))

labs_agg = (
    labs
    .group_by(["subject_id", "hadm_id"]) # Jedna wizyta = jeden wiersz
    .agg(agg_exprs) # Aplikujemy listę wyrażeń (pivotuje tabelę)
    .collect(engine="streaming") 
)

print(f"✅ Zagregowano dane. Wynikowy rozmiar: {len(labs_agg)} wizyt.")
print(f"   Dostępne kolumny: {labs_agg.columns}")


In [ ]:
# --- KONFIGURACJA MAPOWAŃ (8 KLAS) ---

stats = {
    "TARGET_POZ": 0, 
    "TARGET_SPEC_HEMATOLOG": 0, 
    "TARGET_SPEC_NEFROLOG": 0, 
    "TARGET_SPEC_GASTROLOG": 0, 
    "TARGET_SPEC_SOR": 0,
    "TARGET_SPEC_KARDIOLOG": 0,
    "TARGET_SPEC_PULMONOLOG": 0,
    "TARGET_SPEC_HEPATOLOG": 0
}

# 1. Mapowanie ICD -> TARGET (Specjalista)
ICD_TARGET_MAPPING = {
    # NEFROLOGIA (N17-N19, 584-586)
    ('584', '585', 'N17', 'N18', 'N19'): 'TARGET_SPEC_NEFROLOG',
    
    # HEMATOLOGIA (Anemie, krzepliwość)
    ('280', '281', '282', '283', '284', '285', '286', '287', 'D5', 'D6'): 'TARGET_SPEC_HEMATOLOG',
    
    # GASTROLOGIA (Wrzody, krwawienia)
    ('531', '532', '533', '534', 'K25', 'K26', 'K27', 'K28', 'K92'): 'TARGET_SPEC_GASTROLOG',

    # HEPATOLOGIA (Wątroba) - Nowe
    ('571', 'K70', 'K71', 'K72', 'K73', 'K74', 'B18', '070'): 'TARGET_SPEC_HEPATOLOG',

    # KARDIOLOGIA (Serce) - Nowe
    ('410', '411', '412', '413', '414', '428', 'I20', 'I21', 'I25', 'I50'): 'TARGET_SPEC_KARDIOLOG',

    # PULMONOLOGIA (Płuca) - Nowe
    ('490', '491', '492', '493', '496', '480', '481', '482', '483', '484', '485', '486', 'J44', 'J45', 'J18'): 'TARGET_SPEC_PULMONOLOG'
}

# --- STRONG SIGNALS (CLINICAL RULES) ---
# Reguły medyczne z Audytu Jakości Danych + Nowe
STRONG_SIGNALS = {
    'TARGET_SPEC_NEFROLOG': ['CREATININE_Q9', 'CREATININE_Q10', 'UREA_Q9', 'UREA_Q10'],
    'TARGET_SPEC_HEMATOLOG': ['HGB_Q1', 'HGB_Q2', 'PLT_Q1', 'PLT_Q2', 'WBC_Q1', 'WBC_Q10'], 
    'TARGET_SPEC_GASTROLOG': ['SYMPTOM_GI_BLEED_YES', 'SYMPTOM_STOMACH_PAIN_YES'],
    'TARGET_SPEC_SOR': ['_CRITICAL'], 
    
    # NOWE REGUŁY
    'TARGET_SPEC_HEPATOLOG': ['ALT_Q9', 'ALT_Q10', 'AST_Q9', 'AST_Q10', 'SYMPTOM_JAUNDICE_YES'],
    'TARGET_SPEC_KARDIOLOG': ['SYMPTOM_CHEST_PAIN_YES', 'SYMPTOM_EDEMA_YES'], # Obrzęki jako proxy niewydolności serca
    'TARGET_SPEC_PULMONOLOG': ['SYMPTOM_DYSPNEA_YES']
}

# 2. Mapowanie INTENT Pytania -> ICD (Symulacja odpowiedzi TAK)
ICD_QUESTION_MAPPING = {
    "DIABETES_CHECK": ("250", "E10", "E11", "E12", "E13", "E14"),
    "SMOKING_CHECK": ("3051", "Z720", "F17"),
    "ALCOHOL_CHECK": ("303", "3050", "F10", "E244"), 
    "HEAVY_PERIODS": ("6262", "N920", "N921"), 
    "MEDS_CHECK": ("Z9222", "V5861"),
    "GASTRITIS_CHECK": ("535", "K29", "R10"), 
    "GI_BLEED_CHECK": ("578", "K920", "K921", "K922"), 
    "VOMIT_CHECK": ("7870", "R11"), 
    "JAUNDICE_CHECK": ("7824", "R17"), 
    "STOMACH_PAIN_CHECK": ("7890", "R10"), 
    "DYSURIA_CHECK": ("7881", "R30"), 
    "HEMATURIA_CHECK": ("5997", "R31"), 
    "EDEMA_CHECK": ("7823", "R60", "4280", "I50"), 
    "DIET_CHECK": ("26", "E4", "E5", "E6"), 
    "BRUISING_CHECK": ("7827", "R233", "287"), 
    "WEIGHT_LOSS_CHECK": ("7832", "R634"), 
    "CHEST_PAIN_CHECK": ("7865", "R07"), 
    "DYSPNEA_CHECK": ("7860", "R060") 
}

def extract_triggers(tokens):
    triggers = set()
    for t in tokens:
        parts = t.split('_')
        test_name = parts[0]
        if any(x in t for x in ["Q1", "Q2", "Q3", "LOW", "CRITICAL_LOW"]):
            triggers.add(f"{test_name}_LOW")
        if any(x in t for x in ["Q8", "Q9", "Q10", "HIGH", "CRITICAL_HIGH"]):
            triggers.add(f"{test_name}_HIGH")
        if "CRITICAL" in t:
            triggers.add(f"{test_name}_CRITICAL")
    return triggers

# --- START GENEROWANIA (MULTI-LABEL STRATEGY) ---
print("🚀 Generowanie sekwencji MIMIC (Multi-Label Strategy - 8 CLASSES)...")

# Limity (zachowujemy dla kontroli rozmiaru)
CLASS_LIMITS = {
    'TARGET_SPEC_HEMATOLOG': 40000, 
    'TARGET_SPEC_SOR': 40000,
    'TARGET_SPEC_NEFROLOG': 40000,
    'TARGET_SPEC_GASTROLOG': 40000, 
    'TARGET_SPEC_KARDIOLOG': 40000,
    'TARGET_SPEC_PULMONOLOG': 40000,
    'TARGET_SPEC_HEPATOLOG': 40000,
    'TARGET_POZ': 0 
}

print(f"🚀 Generowanie sekwencji MIMIC (Z LIMITAMI KLAS)...\nLimity: {CLASS_LIMITS}")

mimic_sequences = []
stats = {k: 0 for k in CLASS_LIMITS.keys()}
df_visits = labs_agg.sort(["subject_id", "hadm_id"])

# Shuffle
df_visits_shuffled = labs_agg.sample(fraction=1.0, shuffle=True, seed=42)

available_tests = [col for col in df_visits_shuffled.columns if col in MIMIC_MAPPING.values()]

pbar = tqdm(total=len(df_visits_shuffled), desc="Generowanie")

for row in df_visits_shuffled.iter_rows(named=True):
    subj = row["subject_id"]
    
    if subj not in pat_lookup: 
        pbar.update(1)
        continue

    # --- KROK 1: Generowanie Tokenów ---
    age, sex = pat_lookup[subj]
    lab_tokens = []
    
    for test_name in available_tests:
        val = row[test_name]
        if val is not None:
            tok = get_lab_token_v2(test_name, val, age, sex, LAB_NORMS_DB)
            if tok: lab_tokens.append(tok)

    if not lab_tokens:
        pbar.update(1)
        continue

    # --- COMPOSITE FEATURES ---
    def generate_composite_features(lab_tokens):
        composite_tokens = []
        hgb_low = any('HGB_Q1' in t or 'HGB_Q2' in t or 'HGB_Q3' in t for t in lab_tokens)
        mcv_low_normal = any('MCV_Q1' in t or 'MCV_Q2' in t or 'MCV_Q3' in t or 'MCV_Q4' in t or 'MCV_Q5' in t for t in lab_tokens)
        if hgb_low and mcv_low_normal: composite_tokens.append('COMPOSITE_ANEMIA_INDICATOR')
        
        plt_low = any('PLT_Q1' in t or 'PLT_Q2' in t or 'PLT_Q3' in t for t in lab_tokens)
        if plt_low: composite_tokens.append('COMPOSITE_THROMBOCYTOPENIA_INDICATOR')
        
        creatinine_high = any('CREATININE_Q8' in t or 'CREATININE_Q9' in t or 'CREATININE_Q10' in t for t in lab_tokens)
        urea_high = any('UREA_Q8' in t or 'UREA_Q9' in t or 'UREA_Q10' in t for t in lab_tokens)
        if creatinine_high and urea_high: composite_tokens.append('COMPOSITE_RENAL_FAILURE_INDICATOR')
        
        egfr_low = any('EGFR_Q1' in t or 'EGFR_Q2' in t or 'EGFR_Q3' in t for t in lab_tokens)
        if creatinine_high and egfr_low: composite_tokens.append('COMPOSITE_ACUTE_KIDNEY_INJURY')
        
        if plt_low and hgb_low: composite_tokens.append('COMPOSITE_BLEEDING_RISK')
        
        wbc_high = any('WBC_Q8' in t or 'WBC_Q9' in t or 'WBC_Q10' in t for t in lab_tokens)
        if wbc_high: composite_tokens.append('COMPOSITE_INFECTION_INDICATOR')
        
        hgb_critical = any('HGB_Q1_CRITICAL' in t for t in lab_tokens)
        if hgb_critical: composite_tokens.append('COMPOSITE_SEVERE_ANEMIA')
        
        creatinine_critical = any('CREATININE_Q10_CRITICAL' in t for t in lab_tokens)
        if creatinine_critical: composite_tokens.append('COMPOSITE_RENAL_CRITICAL')
        
        # New Composites for new classes?
        # Maybe LIVER_INJURY for ALT/AST high?
        alt_high = any('ALT_Q9' in t or 'ALT_Q10' in t for t in lab_tokens)
        ast_high = any('AST_Q9' in t or 'AST_Q10' in t for t in lab_tokens)
        if alt_high or ast_high:
            composite_tokens.append('COMPOSITE_LIVER_INJURY')

        return composite_tokens
    
    composite_tokens = generate_composite_features(lab_tokens)
    lab_tokens.extend(composite_tokens)

    # --- KROK 2: Symulacja Wywiadu ---
    active_triggers = extract_triggers(lab_tokens)
    tokens = [f"AGE_{int(age/10)*10}", f"SEX_{sex}"] + lab_tokens
    
    age_group = get_age_group(age)
    rules = QUESTIONS_DB.get(age_group, [])
    patient_icds = mimic_icd_map.get(subj, set())
    
    for rule in rules:
        if rule["trigger"] in active_triggers:
            if "gender" in rule and rule["gender"] != sex: continue
            
            intent = rule.get("intent")
            icd_prefixes = ICD_QUESTION_MAPPING.get(intent)
            is_yes = False
            if icd_prefixes:
                for code in patient_icds:
                    if code.startswith(icd_prefixes):
                        is_yes = True
                        break
            answer_token = rule["token_yes"] if is_yes else rule["token_no"]
            tokens.append(answer_token)
            
    # --- KROK 3: MULTI-LABEL ASSIGNMENT ---
    assigned_labels = set()
    
    # A. Labels from ICD
    for code in patient_icds:
        for prefixes, label in ICD_TARGET_MAPPING.items():
            if code.startswith(prefixes):
                assigned_labels.add(label)
                
    # B. Labels from STRONG SIGNALS
    for target_label, indicators in STRONG_SIGNALS.items():
        for ind in indicators:
            if ind.startswith("_"):
                if any(ind in t for t in tokens):
                    assigned_labels.add(target_label)
                    break 
            else:
                if any(ind in t for t in tokens):
                    assigned_labels.add(target_label)
                    break
    
    if not assigned_labels:
        pbar.update(1)
        continue

    # --- KROK 4: Limit Check ---
    needs_save = False
    for l in assigned_labels:
        if stats.get(l, 0) < CLASS_LIMITS.get(l, float('inf')):
            needs_save = True
            break
            
    if not needs_save:
        pbar.update(1)
        continue

    # --- KROK 5: Zapis ---
    for l in assigned_labels:
        if l in stats: stats[l] += 1
        
    label_str = " ".join(sorted(list(assigned_labels)))
    seq = " ".join(tokens) + f" {label_str}"
    mimic_sequences.append(seq)
    pbar.update(1)

pbar.close()
print(f"✅ Wygenerowano {len(mimic_sequences)} sekwencji.")
print("\\n--- FINALNY ROZKŁAD (MIMIC - Multi Label) ---")
for k, v in stats.items():
    print(f"{k} : {v}")


In [ ]:
import paths_config
paths_config.ensure_dirs()
from datetime import datetime
output_file = str(paths_config.DATA_PROCESSED / f"mimic_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
with open(output_file, "w") as f:
    f.write("\n".join(mimic_sequences))

print(f"🎉 Sukces! Zapisano plik: {output_file}")

In [ ]:
# ROZKLAD KLAS
print("\n--- ROZKŁAD KLAS ---")
for label in stats.keys():
    print(f"{label}: {stats[label]}")